# Latent-class logit

Estimate a two-class logit model with class-specific taste coefficients and an estimated class-membership intercept.

This notebook is self-contained and was executed in the repository's Office
validation environment. Install the released package with
`pip install torchdcm`, then select CPU or CUDA through the `device` argument.


In [1]:
import numpy as np
import pandas as pd
import torch

from torchdcm import (
    AlternativeScale,
    Beta,
    ChoiceDataset,
    ChoiceLatentEffect,
    ContinuousIndicator,
    CovariateScale,
    CovariateScaledMultinomialLogit,
    CrossNest,
    CrossNestedLogit,
    ErrorComponent,
    ErrorComponentsLogit,
    HybridChoiceModel,
    LatentClassLogit,
    LatentVariable,
    MixedLogit,
    MultinomialLogit,
    Nest,
    NestedLogit,
    OrderedChoiceDataset,
    OrderedLogit,
    OrderedProbit,
    RandomCoefficient,
    ScaledMultinomialLogit,
    UtilitySpec,
    WTPCoefficient,
    WTPMixedLogit,
)
from torchdcm.datasets import make_swissmetro_like

torch.set_default_dtype(torch.float64)
torch.manual_seed(7)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"TorchDCM example running on {device}")


def show_result(result):
    print(f"Final log likelihood: {result.loglike:.6f}")
    print(f"AIC: {result.aic:.3f}; BIC: {result.bic:.3f}")
    return pd.DataFrame(
        {
            "estimate": result.values,
            "std. error": result.bse,
            "z": result.zvalues,
            "p-value": result.pvalues,
        },
        index=pd.Index(result.param_names, name="parameter"),
    ).round(6)


TorchDCM example running on cuda


In [2]:
def make_choice_data(n_obs=180, seed=7, *, observation_variables=None):
    frame = make_swissmetro_like(n_obs=n_obs, seed=seed)
    data = ChoiceDataset.from_wide(
        frame,
        alternatives=["TRAIN", "SM", "CAR"],
        choice="choice",
        variables={
            "time": {
                "TRAIN": "time_train",
                "SM": "time_sm",
                "CAR": "time_car",
            },
            "cost": {
                "TRAIN": "cost_train",
                "SM": "cost_sm",
                "CAR": "cost_car",
            },
        },
        obs_variables=observation_variables,
        availability={
            "TRAIN": "avail_train",
            "SM": "avail_sm",
            "CAR": "avail_car",
        },
        individual_id="person_id",
    )
    return frame, data


def make_utility_spec(suffix=""):
    tag = f"_{suffix}" if suffix else ""
    spec = UtilitySpec()
    spec.utility(
        "TRAIN",
        Beta(f"ASC_TRAIN{tag}", init=0.0)
        + Beta(f"B_TIME{tag}", init=-0.01) * "time"
        + Beta(f"B_COST{tag}", init=-0.10) * "cost",
    )
    spec.utility(
        "SM",
        Beta(f"B_TIME{tag}", init=-0.01) * "time"
        + Beta(f"B_COST{tag}", init=-0.10) * "cost",
    )
    spec.utility(
        "CAR",
        Beta(f"ASC_CAR{tag}", init=0.0)
        + Beta(f"B_TIME{tag}", init=-0.01) * "time"
        + Beta(f"B_COST{tag}", init=-0.10) * "cost",
    )
    return spec


frame, data = make_choice_data(n_obs=220, seed=20)
print(f"Observations: {data.n_obs}")

Observations: 220


In [3]:
class_specs = [
    make_utility_spec("CLASS_1"),
    make_utility_spec("CLASS_2"),
]
model = LatentClassLogit(
    class_specs,
    class_membership=[Beta("CLASS_2_INTERCEPT", init=0.0)],
    device=device,
    max_iter=60,
)
result = model.fit(data)
show_result(result)


Final log likelihood: -207.433294
AIC: 432.867; BIC: 463.409


/home/baichuan/torchdcm/main-repo/torchdcm/results/result.py:49: RuntimeWarning: divide by zero encountered in divide
  return self.values / self.bse


,estimate,std. error,z,p-value
parameter,,,,
ASC_TRAIN_CLASS_1,-0.380663,1.397092,-0.272468,0.785262
B_TIME_CLASS_1,-0.023180,0.013410,-1.728615,0.083878
B_COST_CLASS_1,-0.089098,0.000000,-inf,0.000000
ASC_CAR_CLASS_1,-0.197143,0.000000,-inf,0.000000
ASC_TRAIN_CLASS_2,-0.380663,1.397092,-0.272468,0.785262
B_TIME_CLASS_2,-0.023180,0.013410,-1.728615,0.083878
B_COST_CLASS_2,-0.089098,0.000000,-inf,0.000000
ASC_CAR_CLASS_2,-0.197143,0.000000,-inf,0.000000
CLASS_2_INTERCEPT,-0.000000,0.000003,-0.000000,1.000000
